In [2]:
from nltk.tokenize import word_tokenize
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

from common.LLMUtils import GPTDatasetv1

In [3]:
with open('dataset/the-verdict.txt', mode='r', encoding='utf-8') as f:
    raw_text = f.read()


In [4]:
all_tokens = word_tokenize(raw_text)

In [5]:
print(type(all_tokens))
print(len(all_tokens))
print(all_tokens[: 10])

<class 'list'>
4544
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']


In [6]:
all_words = sorted(set(all_tokens))
print(len(all_words))
print(all_words[: 10])

1138
['!', "'", "''", "'Are", "'It", "'coming", "'d", "'done", "'ll", "'re"]


In [7]:
vocab = {token:idx for idx, token in enumerate(all_words)}
print(len(vocab))
print(vocab['modesty'])

1138
694


In [8]:
class SimpleTokenier:
    def __init__(self, vocab):
        self.str_to_idx = vocab
        self.idx_to_str = {idx : token for token, idx in vocab.items()}
        print(f'SimpleTokenier init: idx_to_str[\'modesty\']: {self.idx_to_str[694]}')


    def encode(self, text):
        text_tokens = word_tokenize(text)
        ids = [self.str_to_idx[token] for token in text_tokens]
        return ids

    def decode(self, ids):
        text_tokens = [self.idx_to_str[idx] for idx in ids]
        text = " ".join(text_token for text_token in text_tokens)
        return text

In [9]:
tokenizer = SimpleTokenier(vocab)

SimpleTokenier init: idx_to_str['modesty']: modesty


In [10]:
ids = tokenizer.encode('I glanced after him, struck by his last word.')
print(ids)

[64, 513, 164, 563, 17, 956, 266, 566, 618, 1125, 19]


In [11]:
text = tokenizer.decode([64, 513, 164, 563, 17, 956, 266, 566, 618, 1125, 19])
print(text)

I glanced after him , struck by his last word .


In [12]:
# add <|endoftext|> <|unk|>
class SimpleTokenierv2:
    def __init__(self, vocab):
        self.str_to_idx = vocab
        self.idx_to_str = {idx : token for token, idx in vocab.items()}
        print(f'SimpleTokenier init: idx_to_str[\'modesty\']: {self.idx_to_str[694]}')


    def encode(self, text):
        text_tokens = word_tokenize(text)
        ids = []
        for token in text_tokens:
            if token in self.str_to_idx:
                ids.append(self.str_to_idx[token])
            else:
                ids.append(self.str_to_idx['<|unk|>'])
        return ids

    def decode(self, ids):
        text_tokens = [self.idx_to_str[idx] for idx in ids]
        text = " ".join(text_token for text_token in text_tokens)
        return text

In [13]:
all_words = sorted(set(all_tokens))
all_words.extend(['<|endoftext|>', '<|unk|>'])
print(type(all_words))
print(len(all_words))
print(all_words[: 10])

<class 'list'>
1140
['!', "'", "''", "'Are", "'It", "'coming", "'d", "'done", "'ll", "'re"]


In [14]:
vocab = {token:idx for idx, token in enumerate(all_words)}
print(len(vocab))
print(vocab['<|endoftext|>'])
print(vocab['<|unk|>'])

1140
1138
1139


In [15]:
tokenizerv2 = SimpleTokenierv2(vocab)
ids = tokenizerv2.encode('I glanced after Xuejian Li, struck by his last word.')
print(ids)

SimpleTokenier init: idx_to_str['modesty']: modesty
[64, 513, 164, 1139, 1139, 17, 956, 266, 566, 618, 1125, 19]


In [16]:
tiktokenTokenier = tiktoken.get_encoding('gpt2')

In [17]:
ids = tiktokenTokenier.encode('Hello world, <|endoftext|> Xuejian', allowed_special={"<|endoftext|>"})
print(ids)

[15496, 995, 11, 220, 50256, 47298, 73, 666]


In [18]:
with open('dataset/the-verdict.txt', mode='r', encoding='utf-8') as f:
    raw_text = f.read()
max_len = 4
batch_szie = 1
stride = 1
datasetv1 = GPTDatasetv1(raw_text, max_len, stride)
dataloaderv1 = DataLoader(
    datasetv1,
    batch_size=batch_szie,
    shuffle=True,
    drop_last=True,
    num_workers=1,
)

GPTDatasetv1 init completed


In [19]:
data_iter = iter(dataloaderv1)
first_batch = next(data_iter)
print(first_batch)

[tensor([[ 3474.,   416.,  1310., 40559.]]), tensor([[  416.,  1310., 40559., 11959.]])]


In [20]:
vocab_size = 6
input_dim = 3
embedding_layer = torch.nn.Embedding(vocab_size, input_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[-1.0332,  0.2437, -0.0699],
        [-0.3012, -1.8246, -0.1050],
        [-1.0257,  0.7027, -0.1809],
        [ 1.4813, -0.2542,  0.1513],
        [-1.7592,  1.4113, -0.7947],
        [-1.7528, -1.8434,  2.0496]], requires_grad=True)
